# PWM scanning: how many binding sites does a motif predict?

## 1. False positives of GAL4 in Yeast

Create a directory and go in it

In [1]:
mkdir -p pwm_scanning
cd pwm_scanning

load all the necessary tools (datacore) and check their availability

In [2]:
module load MEME/5.5.5-gompi-2023a
module load SAMtools/1.18-GCC-12.3.0
module load BEDTools/2.31.0-GCC-12.3.0

which fimo jaspar2meme fasta-get-markov fasta-shuffle-letters bedtools samtools
fimo --version

/mnt/modules/easybuild/software/MEME/5.5.5-gompi-2023a/bin/fimo
/mnt/modules/easybuild/software/MEME/5.5.5-gompi-2023a/libexec/meme-5.5.5/jaspar2meme
/mnt/modules/easybuild/software/MEME/5.5.5-gompi-2023a/libexec/meme-5.5.5/fasta-get-markov
/mnt/modules/easybuild/software/MEME/5.5.5-gompi-2023a/libexec/meme-5.5.5/fasta-shuffle-letters
/mnt/modules/easybuild/software/BEDTools/2.31.0-GCC-12.3.0/bin/bedtools
/mnt/modules/easybuild/software/SAMtools/1.18-GCC-12.3.0/bin/samtools
5.5.5


We download the yeast genome (sacCer3) from UCSC

In [3]:
wget https://hgdownload.soe.ucsc.edu/goldenPath/sacCer3/bigZips/sacCer3.fa.gz
gunzip -c sacCer3.fa.gz > sacCer3_all.fa

--2026-10-30 19:53:20--  https://hgdownload.soe.ucsc.edu/goldenPath/sacCer3/bigZips/sacCer3.fa.gz


Resolving hgdownload.soe.ucsc.edu (hgdownload.soe.ucsc.edu)... 128.114.119.163
Connecting to hgdownload.soe.ucsc.edu (hgdownload.soe.ucsc.edu)|128.114.119.163|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3820548 (3.6M) [application/x-gzip]
Saving to: ‘sacCer3.fa.gz’

sacCer3.fa.gz       100%[===================>]   3.64M  3.51MB/s    in 1.0s    

2026-10-30 19:53:24 (3.51 MB/s) - ‘sacCer3.fa.gz’ saved [3820548/3820548]



Remove mitochondrial DNA from yeats genome (distorts background)

In [4]:
samtools faidx sacCer3_all.fa
samtools faidx sacCer3_all.fa $(cut -f 1 sacCer3_all.fa.fai | grep -v chrM) > yeast.fa
samtools faidx yeast.fa
cut -f 1,2 yeast.fa.fai > yeast.chromsizes
cat yeast.chromsizes

chrI	230218
chrII	813184
chrIII	316620
chrIV	1531933
chrIX	439888
chrV	576874
chrVI	270161
chrVII	1090940
chrVIII	562643
chrX	745751
chrXI	666816
chrXII	1078177
chrXIII	924431
chrXIV	784333
chrXV	1091291
chrXVI	948066


Size of the yeast genome (but ignore Ns)

In [5]:
grep -v ">" yeast.fa | tr -d '\nNn' | wc -c

12071326


Get yeast genomic background

In [6]:
fasta-get-markov -m 0 yeast.fa yeast.bg > /dev/null 2>&1
cat yeast.bg

# 0-order Markov frequencies from file yeast.fa
# seqs: 16    min: 230218    max: 1531933    avg: 754457.9    sum: 12071326    alph: DNA
# order 0
A 3.085e-01
C 1.915e-01
G 1.915e-01
T 3.085e-01


Now we already have some **information** per basepair. If we were to make a guess at each basepair if it was an A,C,G or T, then guessing A or T would be correct 30.9% of the time instead of the expected 25%.

This reduces the amount of yes or no questions we need to ask to make a prediction. In a perfectly balanced genome where each basepair occurs at 1/4 of positions, we need to ask two questions: 1. is it a weak (A/T) or strong (G/C) nucleotide? 2. is it A or T | is it G or C? Each question gives **1 bit** of information.

In the yeast genome, we have to still ask these questions, but the one about strong vs weak answers less than 1 bit, since the answer is 'weak' 61.8% of the time. To really quantify how much information we get per yes or no questions, we can take the sum of the -log<sub>2</sub> of the frequencies of the possibilities scaled by the frequencies themselves:

H = −Σ f<sub>b</sub> log<sub>2</sub>(f<sub>b</sub>)

This is called **entropy**. We take the -log<sub>2</sub> because we work with bits, which have only **2** possible values. If we calculate the entropy for each basepair of the yeast genome, we get:

H = −0.309 log<sub>2</sub> 0.309 - 0.191 log<sub>2</sub> 0.191 − 0.309 log<sub>2</sub> 0.309 - 0.191 log<sub>2</sub> 0.191 ≈ 1.96 bits

So we still need 1.96 bits of information, i.e. we only know 0.04 bits of information.

Try to calculate it yourself:


In [7]:
awk '/^[ACGT] / {H -= $2 * log($2) / log(2)} END {print "H =", H, "bits"}' yeast.bg

H = 1.96013 bits


### Yeast genes and the known GAL4 targets

The gene table `sgdGene` from UCSC is converted to a BED file with the systematic name and the common gene name.

In [8]:
wget https://hgdownload.soe.ucsc.edu/goldenPath/sacCer3/database/sgdGene.txt.gz
zcat sgdGene.txt.gz > sgdGene.txt
head -n 2 sgdGene.txt

--2026-10-30 19:54:34--  https://hgdownload.soe.ucsc.edu/goldenPath/sacCer3/database/sgdGene.txt.gz
Resolving hgdownload.soe.ucsc.edu (hgdownload.soe.ucsc.edu)... 128.114.119.163
Connecting to hgdownload.soe.ucsc.edu (hgdownload.soe.ucsc.edu)|128.114.119.163|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 133860 (131K) [application/x-gzip]
Saving to: ‘sgdGene.txt.gz’

sgdGene.txt.gz      100%[===================>] 130.72K   441KB/s    in 0.3s    

2026-10-30 19:54:36 (441 KB/s) - ‘sgdGene.txt.gz’ saved [133860/133860]

585	YAL069W	chrI	+	334	649	334	649	1	334,	649,	n/a
585	YAL068W-A	chrI	+	537	792	537	792	1	537,	792,	n/a


make a cleaned up bed file: chrom start end name . strand

In [9]:
awk 'BEGIN{OFS="\t"} {print $3, $5, $6, $2, ".", $4}' sgdGene.txt | sort -k1,1 -k2,2n > genes.bed
head -n 3 genes.bed

chrI	334	649	YAL069W	.	+
chrI	537	792	YAL068W-A	.	+
chrI	1806	2169	YAL068C	.	-


Ren et al. (2000) found only about ten direct targets genes which promoters have real GAL4 binding sites. We put them in a file.

In [10]:
cat > gal4_targets.txt << EOF
YBR020W
YBR019C
YBR018C
YLR081W
YML051W
YDR009W
YOR120W
YDR277C
YGL134W
YBR021W
EOF

Steal them from the cleaned up gene.bed file

In [11]:
grep -wFf gal4_targets.txt genes.bed > gal4_target_genes.bed

Demarcate promotors as [TSS - 800, TSS]

In [12]:
bedtools flank -i gal4_target_genes.bed -g yeast.chromsizes -l 800 -r 0 -s > gal4_promoters.bed
cat gal4_promoters.bed

chrII	275527	276327	YBR018C	.	-
chrII	278352	279152	YBR019C	.	-
chrII	278220	279020	YBR020W	.	+
chrII	280642	281442	YBR021W	.	+
chrIV	462633	463433	YDR009W	.	+
chrIV	1015702	1016502	YDR277C	.	-
chrVII	254862	255662	YGL134W	.	+
chrXII	289411	290211	YLR081W	.	+
chrXIII	170793	171593	YML051W	.	+
chrXV	550313	551113	YOR120W	.	+


### PWM time!

Go to Jaspar: https://jaspar.elixir.no/

Search for GAL4: https://jaspar.elixir.no/search?q=GAL4&collection=all&tax_group=all&tax_id=all&type=all&class=all&family=all&version=all

Download the GAL4 PWM in meme format: https://jaspar.elixir.no/api/v1/matrix/MA0299.1.meme

In [13]:
wget https://jaspar.elixir.no/api/v1/matrix/MA0299.1.meme

--2026-10-30 19:54:51--  https://jaspar.elixir.no/api/v1/matrix/MA0299.1.meme


Resolving jaspar.elixir.no (jaspar.elixir.no)... 158.37.65.54
Connecting to jaspar.elixir.no (jaspar.elixir.no)|158.37.65.54|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 833 [text/meme]
Saving to: ‘MA0299.1.meme’

MA0299.1.meme       100%[===================>]     833  --.-KB/s    in 0s      

2026-10-30 19:54:51 (41.5 MB/s) - ‘MA0299.1.meme’ saved [833/833]



If we were to scan the genome, how many false positives do we get, given this PWM?

* **R<sub>frequency</sub>** = log<sub>2</sub>(#bp<sub>genome</sub> / γ)
* **R<sub>sequence</sub>** = Σ f<sub>i</sub> log<sub>2</sub>(f<sub>i</sub> / background)

where

* γ = #instances<sub>real</sub>
* f<sub>i</sub> = frequency of the basepair at position i in the PWM

A random position matches the motif about once every 2<sup>R<sub>sequence</sub></sup> positions, which predicts roughly **2<sup>(R<sub>frequency</sub> − R<sub>sequence</sub>)</sup> matches per real site**.

In [14]:
# R_sequence: sum over all positions and bases of f * log2(f / background)
RSEQ=$(awk 'NR==FNR {bg[$1] = $2; next}
            /letter-probability/ {m = 1; next}
            m && NF == 4 {split("A C G T", b, " "); for (i = 1; i <= 4; i++) if ($i > 0) RSEQ += $i * log($i / bg[b[i]]) / log(2)}
            END {print RSEQ}' yeast.bg MA0299.1.meme)

# R_frequency: log2(genome size / number of real sites)
GAMMA=20
L_YEAST=$(grep -v ">" yeast.fa | tr -d '\nNn' | wc -c)
RFREQ=$(awk -v G=$L_YEAST -v g=$GAMMA 'BEGIN {print log(G / g) / log(2)}')

echo "R_sequence  = $RSEQ bits"
echo "R_frequency = $RFREQ bits"
echo "expected matches per real site = $(awk -v rs=$RSEQ -v rf=$RFREQ 'BEGIN {print 2^(rf - rs)}')"

# Yes this code was AI generated

R_sequence  = 13.4731 bits
R_frequency = 19.2032 bits
expected matches per real site = 53.0801


R<sub>sequence</sub> is the average GAL4 sequence score: 13.5 bits. But what about single sequences? As an example of how motifs scan the genome, lets score this sequence: CGGGGGACAGTGCTC, which you can already see matches the PWM quite well. That is because it is the consensus sequence, the best matching sequence. At each position we add log<sub>2</sub>(f / background) for the base that is there. Position 1 is a C, and all 2779 reference sites have a C there, so f = 1 and the contribution is log<sub>2</sub>(1 / 0.191) = 2.39 bits. Sum this across all bases and you get 20.9 bits.

We can also derive a p-value: the probability that a random stretch of yeast DNA of the same length scores at least as high. The previous sequence is the highest scoring one and the chance of hitting it is:

p = 0.191<sup>11</sup> × 0.309<sup>4</sup> ≈ 1.1 × 10<sup>−10</sup>

For a score S of, for example, 10 bits, the p-value is found by taking all possible 15-bp sequences that score at least 10 bits and summing their background probabilities.

There is also a quick rule of thumb. A score of S bits means the sequence is 2<sup>S</sup> times more likely under the motif than in random DNA. So random DNA can reach a score of S bits with a probability of at most 2<sup>−S</sup>:

p ≤ 2<sup>−S</sup>

For the average GAL4 sequence that is p = 2<sup>-13.5</sup> =  0.0000881. If we only accepted sites scoring at least the average of 13.5 bits, we would keep about half of the true binding sites. A FIMO threshold of p < 10<sup>−4</sup> is more lenient: it corresponds to a score of about 10.4 bits and keeps around 85% of the sites the matrix describes. The price is that it also lets in more chance hits: about 2 × L × 10<sup>−4</sup> ≈ 2,400 in the yeast genome.

We run FIMO with both GAL4 matrices on the whole yeast genome and keep every hit with p < 10<sup>-4</sup>

`--text` writes all hits straight to the output

In [15]:
for m in MA0299.1 GAL4_ext; do
  fimo --text --thresh 1e-4 --bfile yeast.bg --motif $m MA0299.1.meme yeast.fa 2> /dev/null > fimo_yeast_$m.tsv
done

How many hits do we get?

In [16]:
wc -l fimo_yeast_MA0299.1.tsv

2014 fimo_yeast_MA0299.1.tsv


remember we predicted 2400, so not bad.

## 2. CEBPA on the human genome in liver

CEBPA (C/EBPα) is a bZIP factor and a master regulator of liver genes. It binds DNA as a dimer; its JASPAR matrix MA0102.3 (from ChIP-seq) is 11 bp long and carries ~13 bits, about as much as GAL4. Is that enough for a human chromosome? Chromosome 19 alone (~58 Mb) is almost five times the size of the whole yeast genome.

In [38]:
samtools faidx ../genome/hg38.fa chr19 > chr19.fa
samtools faidx chr19.fa
cut -f 1,2 chr19.fa.fai > chr19.chrom.sizes

None-N basepairs in human chr19

In [19]:
grep -v ">" chr19.fa | tr -d '\nNn' | wc -c

58440758


Remember how we made a background frequency composition file for yeast? Do the same thing for human chromosome 19:

In [20]:
fasta-get-markov -m 0 chr19.fa chr19.bg > /dev/null 2>&1
cat chr19.bg

# 0-order Markov frequencies from file chr19.fa
# seqs: 1    min: 58617616    max: 58617616    avg: 58617616.0    sum: 58617616    alph: DNA
# order 0
A 2.603e-01
C 2.397e-01
G 2.397e-01
T 2.603e-01


On JASPAR, find the meme file for the MA0102.3 human CEBPA ChIP-seq derived PWM.

In [21]:
wget https://jaspar.elixir.no/api/v1/matrix/MA0102.3.meme

--2026-10-30 20:35:12--  https://jaspar.elixir.no/api/v1/matrix/MA0102.3.meme


Resolving jaspar.elixir.no (jaspar.elixir.no)... 158.37.65.54
Connecting to jaspar.elixir.no (jaspar.elixir.no)|158.37.65.54|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 675 [text/meme]
Saving to: ‘MA0102.3.meme’

MA0102.3.meme       100%[===================>]     675  --.-KB/s    in 0s      

2026-10-30 20:35:13 (15.9 MB/s) - ‘MA0102.3.meme’ saved [675/675]



Now fimo the living f*** out of chromosome 19 usign this PWM

In [22]:
fimo --text --thresh 1e-4 --bfile chr19.bg --motif MA0102.3 MA0102.3.meme chr19.fa 2> /dev/null > fimo_cebpa.tsv

clean up the output

In [23]:
awk 'BEGIN{OFS="\t"} NR > 1 {print $3, $4-1, $5, $8, $7, $6}' fimo_cebpa.tsv | sort -k1,1 -k2,2n > cebpa_hits.bed
wc -l cebpa_hits.bed
head -n 3 cebpa_hits.bed

16246 cebpa_hits.bed
chr19	61176	61187	6.67e-06	14.6531	+
chr19	65854	65865	9.24e-05	10.898	+
chr19	67424	67435	6.93e-05	11.4082	+


How many hits do we expect by chance? FIMO tests 2 × L windows (both strands), and each passes the threshold by chance with probability p.

In [ ]:
L_CHR=$(grep -v ">" chr19.fa | tr -d '\nNn' | wc -c)
awk -v L=$L_CHR -v p=1e-4 'BEGIN {printf "expected strand hits: %.0f\n", 2 * L * p}'
wc -l < cebpa_hits.bed

expected strand hits: 11688
16246


### CEBPA in peaks

create a neat bed file of ChIP peaks

In [25]:
grep -w chr19 ../res/CEBPA_peaks.filtered.narrowPeak | cut -f 1-3 > cebpa_peaks.bed
wc -l cebpa_peaks.bed

371 cebpa_peaks.bed


Use bedtools intersect to find the hits in peaks

In [31]:
bedtools intersect -u -a cebpa_hits.bed -b cebpa_peaks.bed > sites_in_peaks.bed
wc -l < sites_in_peaks.bed

162


Now the peaks that have hits

In [32]:
bedtools intersect -u -a cebpa_peaks.bed -b cebpa_hits.bed > peaks_with_site.bed
wc -l < peaks_with_site.bed

143


A site outside our peaks is not necessarily false. These peaks come from **one** ChIP-seq experiment in **liver**. CEBPA is also active in fat tissue and in blood cells, and other experiments might find binding at some of these sites. We can never prove that a site is *never* used.

Let's investigate what makes bound sites different from unbound sites. Take all strong chr19 p<1e-5 hits.

In [110]:
awk '$4 <= 1e-5' cebpa_hits.bed > strong_sites.bed

Separate into strong & bound and strong & unbound (not in ChIP peaks)

In [111]:
bedtools intersect -u -a strong_sites.bed -b cebpa_peaks.bed > strong_bound.bed
bedtools intersect -v -a strong_sites.bed -b cebpa_peaks.bed > strong_unbound.bed
wc -l strong_bound.bed strong_unbound.bed

   38 strong_bound.bed
 1128 strong_unbound.bed
 1166 total


Let's be fancy shmancy and calculate the mean GC content around hits and the standard error

In [112]:
for f in strong_bound strong_unbound; do
  awk -v f=$f '{s += $1; ss += $1*$1; n++}
               END {m = s/n; sd = sqrt(ss/n - m*m); printf "%s: mean GC = %.3f, se = %.4f (%d sites)\n", f, m, sd/sqrt(n), n}' $f.gc.txt
done

strong_bound: mean GC = 0.493, se = 0.0129 (38 sites)
strong_unbound: mean GC = 0.419, se = 0.0026 (1128 sites)


49.3% and 41.9% GC might look similar but this is actually quite the difference. GC rich areas are easier to bend and wrap around nucleosomes. This **might** mean that evolution has given this site high GC to make it harder to access unless there is CEBP and therefor maybe more specific to liver tissues. Other sites next to CEBP might then bind to cofactors that initiate transcription. Im speculating here.

Let's see if this holds any truth. We will check if other canonical Hepatocyte binding sites are enriched in ChIP peak CEBPA hits. Just run the following code without thinking about it. The binding sites we test for are MA0114.5 (HNF4A), MA0047.4 (FOXA2), MA0046.3 (HNF1A) and MA0679.3 (ONECUT1)

In [116]:
N=$(wc -l < strong_bound.bed)   # all bound sites
NU=100 #$(wc -l < strong_bound.bed)                           # top unbound sites
sort -k4,4g strong_bound.bed   | head -n $N  | sort -k1,1 -k2,2n > top_bound.bed
sort -k4,4g strong_unbound.bed | head -n $NU | sort -k1,1 -k2,2n > top_unbound.bed
for f in top_bound top_unbound; do
  bedtools slop -b 100 -i $f.bed -g chr19.chrom.sizes > $f.win.bed
done

In [117]:
for id in MA0114.5 MA0047.4 MA0046.3 MA0679.3; do
  bedtools intersect -c -a top_bound.win.bed   -b $id.hits.bed | awk '{print $NF}' > $id.bound.counts
  bedtools intersect -c -a top_unbound.win.bed -b $id.hits.bed | awk '{print $NF}' > $id.unbound.counts
done

In [122]:
module load R      # if R is not available yet; use the module name 'module avail R' shows
for id in MA0114.5 MA0047.4 MA0046.3 MA0679.3; do
  Rscript -e "b <- scan('$id.bound.counts', quiet = TRUE); u <- scan('$id.unbound.counts', quiet = TRUE)
    p <- wilcox.test(b, u, exact = FALSE)\$p.value
    cat('$id', ' bound: total', sum(b), '/', '$N', ' mean', round(mean(b), 2),
        '| unbound: total', sum(u), '/', '$NU', ' mean', round(mean(u), 2), '| p =', signif(p, 2), '\n')"
done

MA0114.5  bound: total 6 / 38  mean 0.16 | unbound: total 2 / 100  mean 0.02 | p = 0.0077 
MA0047.4  bound: total 2 / 38  mean 0.05 | unbound: total 7 / 100  mean 0.07 | p = 0.86 
MA0046.3  bound: total 5 / 38  mean 0.13 | unbound: total 5 / 100  mean 0.05 | p = 0.23 
MA0679.3  bound: total 4 / 38  mean 0.11 | unbound: total 5 / 100  mean 0.05 | p = 0.24 


Cool! Looks like HNF4 has more hits within +-50bps of bound CEBPA sites! It is not known which one has more of a pioneer function/ chromatine remodeloing function/ transcription activation function. But cool nonetheless. 

Another factor that might be different between bound and unbound hits is the proximity of other significant hits. When there are more binding sites closeby, thermodynamically speaking, there will be more CEBPA binding molecules in the vacinity in the nucleus. This is called clustering. Lets see if clustering is more common in ChIP bound sites than in unbound sites.

In [165]:
bedtools makewindows -g chr19.chrom.sizes -w 500 -s 250 > win500.bed
bedtools intersect -c -a win500.bed -b hits_1e-4.bed > win500_counts.bed

In [166]:
echo -e "min_sites\tregions\tin_peak\tfraction"
for k in 1 2 3 4 5; do
  awk -v k=$k '$4 >= k' win500_counts.bed | bedtools merge > clusters_k.bed
  n=$(wc -l < clusters_k.bed)
  inpk=$(bedtools intersect -u -a clusters_k.bed -b cebpa_peaks.bed | wc -l)
  awk -v k=$k -v n=$n -v i=$inpk 'BEGIN {printf "%d\t%d\t%d\t%.3f\n", k, n, i, (n > 0) ? i/n : 0}'
done

min_sites	regions	in_peak	fraction


1	12083	172	0.014
2	1702	36	0.021
3	127	5	0.039
4	8	1	0.125
5	1	0	0.000


Great! for k=1,2,3 & 4, the fraction of ChIP sites increases against other hits. This means clustering happens more in 'true' regions. Lets calculate a p value for this though:

In [168]:
awk '$4 >= 1' win500_counts.bed | bedtools merge > clusters_1.bed
awk '$4 >= 2' win500_counts.bed | bedtools merge > clusters_2.bed
bedtools intersect -v -a clusters_1.bed -b clusters_2.bed > single_site_regions.bed     # regions with only 1 site

a=$(bedtools intersect -u -a clusters_2.bed -b cebpa_peaks.bed | wc -l); A=$(wc -l < clusters_2.bed)
b=$(bedtools intersect -u -a single_site_regions.bed -b cebpa_peaks.bed | wc -l); B=$(wc -l < single_site_regions.bed)
Rscript -e "t <- matrix(c($a, $A-$a, $b, $B-$b), 2); print(t); cat('Fisher p =', signif(fisher.test(t)\$p.value, 2), '\n')"

     [,1]  [,2]
[1,]   36   130
[2,] 1666 10281
Fisher p = 0.0067 


Congrats! We are slowly unraveling enhancer grammar!